# Demagnetization field

Let us assume a magnetization of the form
$$\mathbf{M}=M_s \mathbf{m},$$
nonzero only inside a magnetic region $\Omega_m$.

This magnetization creates a magnetic field $\mathbf{H}^{\mathsf{dem}}$ on $\mathbb{R}^3$ satisfying
$$
\nabla \cdot (\mathbf{H}^{\mathsf{dem}} + \mathbf{M}) = 0.
$$

The magnetic field also satisfies
$$
\nabla \times \mathbf{H}^{\mathsf{dem}} = 0,
$$
thus it is a conservative field and admits a scalar potential $u$ satisfying
$$
\mathbf{H}^{\mathsf{dem}} = - \nabla u.
$$

The scalar field $u$ satisfies the Poisson's equation
$$
\Delta u = \nabla \cdot \mathbf{M}
$$
with open boundary conditions (i.e. $u$ is zero at infinity).

In this notebook we will evaluate the demagnetization field for $\Omega_m$ a cube of side length 50, spontaneous magnetization $M_s=1.76$, and $\mathbf{m}=(0,0,1)$.

## Truncation approach and weak formulation

To solve this problem we use the so-called truncation approach, or airbox method.

We define a domain $\Omega$ big enough (such that $\Omega_m \subset \Omega$) and look for $u$ solution of
$$
\left\{
\begin{array}{ll}
    - \Delta u = - \nabla \cdot M_s \mathbf{m} & \text{in } \Omega, \\
    u = 0 & \text{on } \partial \Omega.
\end{array}\right.
$$
where
$$
M_s(\mathbf{x}) := \left\{
\begin{array}{ll}
    1.76 & \mathbf{x} \in \Omega_m, \\
    0 & x \in \partial \Omega - \Omega_m.
\end{array}\right.
$$
and $\mathbf{m}=(0,0,1)$ constant.

We define the solution space
$$
V_0 := \{ v \in H^1(\Omega) : \, \left.v\right|_{\partial\Omega}=0 \}.
$$

Then we weak formulation of the problem reads: find $u \in V_0$ such that
$$
\int_\Omega \nabla u (\mathbf{x}) \cdot \nabla v (\mathbf{x}) \ \mathrm{d}\mathbf{x} = M_s \int_{\Omega_m} \mathbf{m} \cdot \nabla v (\mathbf{x}) \ \mathrm{d}\mathbf{x} \qquad \forall \, v \in V_0.
$$

## Loading the mesh

In [1]:
import skfem
from skfem.helpers import grad, dot
from skfem.models.poisson import mass
from skfem.visuals.matplotlib import draw
import numpy as np

In [2]:
mesh = skfem.Mesh.load("mesh/Omega.msh")
mesh

<skfem MeshTet1 object>
  Number of elements: 291713
  Number of vertices: 48927
  Number of nodes: 48927
  Named subdomains [# elements]: core [71373], shell [220340]
  Named boundaries [# facets]: None [1478]

In [3]:
# draw(mesh)

## Solving the PDE

In [4]:
basis_1 = skfem.Basis(mesh, skfem.ElementTetP1())

In [5]:
core_basis = skfem.Basis(mesh, basis_1.elem, elements=mesh.subdomains['core'])

In [6]:
@skfem.BilinearForm
def stiffness(u, v, _):
    return dot(grad(u), grad(v))
S = stiffness.assemble(basis_1)

As the scalar $M_s$ and the vector $\mathbf{m}$ are constant on each magentic material $\Omega_i$
$$
M_s(\mathbf{x}) = M_s^i
\quad
\text{and}
\quad
\mathbf{m}(\mathbf{x}) =
\left[
\begin{array}{c}
m^i_x\\
m^i_y\\
m^i_z
\end{array}
\right] \qquad \text{for } \mathbf{x} \in \Omega_i,
$$
we can in general split the right-hand side
$$
int_{\Omega_m} M_s (\mathbf{x}) \mathbf{m}(\mathbf{x}) \cdot \nabla v (\mathbf{x}) \ \mathrm{d}\mathbf{x}
$$
as
$$
\sum_i M_s^i \left[
m^i_x \int_{\Omega_i} \frac{\partial v}{\partial x}(\mathbf{x}) \ \mathrm{d}\mathbf{x}
+ m^i_y \int_{\Omega_i} \frac{\partial v}{\partial y}(\mathbf{x}) \ \mathrm{d}\mathbf{x}
+ m^i_z \int_{\Omega_i} \frac{\partial v}{\partial z}(\mathbf{x}) \ \mathrm{d}\mathbf{x}
\right]
$$
and discretize as
$$
\sum_i M_s^i \left( m^i_x B^i_x + m^i_y B^i_y + m^i_z B^i_z \right)
$$
with reasonable matrices $B^i_x, B^i_y, B^i_z$.

In [7]:
Ms = 14e5
m_array = [0, 0, 1]
b = 0
for x, m in enumerate(m_array):
    @skfem.LinearForm
    def B(v, w):
        return grad(v)[x]
    b += Ms * m * B.assemble(core_basis)

In [8]:
b

array([ 1366494.84079523, -1366494.84079523,  1366494.84079524, ...,
              0.        ,        0.        ,        0.        ])

In [9]:
%%time
y = skfem.solve(*skfem.condense(S, b, D=mesh.boundary_nodes()))

CPU times: user 5min 52s, sys: 2.85 s, total: 5min 55s
Wall time: 1min 19s


In [10]:
mesh.save("solution/u.vtk", point_data={"u": y})

We can alternatively solve

In [11]:
%%time
from scipy.sparse.linalg import spsolve

S_cnd, rhs_cnd, _, sol_indices = skfem.condense(A=S, b=b, D=mesh.boundary_nodes())
sol = spsolve(A=S_cnd, b=rhs_cnd)

CPU times: user 5min 52s, sys: 3.47 s, total: 5min 55s
Wall time: 1min 17s


`sol` is the solution only defined on some indices: if we had
```python
u = basis_1.zeros()
u[sol_indices] = sol
```
then we would have `y=u`.

In [12]:
sol.shape

(44496,)

In [13]:
y.shape

(48927,)

## Calculate the $\mathbf{H}^\mathrm{dem}$ from the scalar potential

Project the gradient of the scalar potential to the vector function space on which magnetisation is defined.

In [14]:
ev = skfem.ElementVectorH1(skfem.ElementTetP1())
H1P0 = skfem.Basis(mesh, ev)
H1P0

<skfem CellBasis(MeshTet1, ElementVector) object>
  Number of elements: 291713
  Number of DOFs: 146781
  Size: 1344213504 B

In [15]:
%%time
H_dem = H1P0.project(-grad(basis_1.interpolate(y)))

CPU times: user 14min 46s, sys: 21.8 s, total: 15min 8s
Wall time: 5min 50s


In [16]:
H_dem.shape

(146781,)

In [17]:
mesh.save("solution/H_dem.vtk", {"H": H_dem[H1P0.nodal_dofs].T})

## Demagnetization energy

The energy connected to the demagnetization field is given by (TO CHECK)
$$
E^{\mathsf{dem}} = - \frac{\mu_0}{2} \int_\Omega M_s(\mathbf{x}) \, \mathbf{m} (\mathbf{x}) \cdot \mathbf{H}^{\mathsf{dem}} (\mathbf{x}) \ \mathrm{d}\mathbf{x}.
$$
In this case,
$$
E^{\mathsf{dem}} = - \frac{\mu_0}{2} M_s \int_{\Omega_m} H_z^{\mathsf{dem}} (\mathbf{x}) \ \mathrm{d}\mathbf{x}.
$$


We extract the function $H^{\mathsf{dem}}_z$ on the subdomain `core`:

In [29]:
Hz = core_basis.interpolate(H_dem[H1P0.nodal_dofs][2])
Hz

<skfem DiscreteField object>
  Quadrature points per element: 4
  Number of elements: 71373
  Order: 0
  Attributes: grad

And we evaluate the formula

In [30]:
from scipy.constants import mu_0
@skfem.Functional
def integral(w):
    return w["Hz"]
E_dem = - 0.5 * mu_0 * Ms * integral.assemble(core_basis, Hz=Hz)
E_dem

48458847875.32336

## Solution with `mammos-mumag`

Compare demag energy

In [31]:
import mammos_entity as me
from mammos_mumag.materials import Materials
from mammos_mumag.parameters import Parameters
from mammos_mumag.simulation import Simulation
sim = Simulation(
    mesh="cube50_singlegrain_msize2",
    materials=Materials(
        domains=[
            {
                "theta": 0.0,
                "phi": 0.0,
                "K1": 0.0,
                "K2": 0.0,
                "Ms": me.Ms(Ms),
                "A": 0.0,
            },
            {
                "theta": 0.0,
                "phi": 0.0,
                "K1": 0.0,
                "K2": 0.0,
                "Ms": 0.0,
                "A": 0.0,
            },
            {
                "theta": 0.0,
                "phi": 0.0,
                "K1": 0.0,
                "K2": 0.0,
                "Ms": 0.0,
                "A": 0.0,
            },
        ],
    ),
    parameters=Parameters(
        size=1.0e-9,
        scale=0,
        m_vect=[0, 0, 1],
        hstart=1,
        hfinal=-1,
        hstep=-0.2,
        h_vect=[0.01745, 0, 0.99984],
        mstep=0.4,
        mfinal=-1.2,
        tol_fun=1e-10,
        tol_hmag_factor=1,
        precond_iter=10,
    ),
)

In [32]:
%%time
sim.run_hmag(outdir="mammos-mumag-hmag", name="cube")

CPU times: user 76.1 ms, sys: 68.3 ms, total: 144 ms
Wall time: 59.4 s


In [33]:
import pandas as pd
pd.read_csv("mammos-mumag-hmag/cube.csv", skiprows=1)

,name,value,explanation
0,E_field,399196.443355,Energy density evaluated from field (J/m^3).
1,E_gradient,399196.443355,Energy density evaluated from gradient (J/m^3).
2,E_analytic,410501.440570,Energy density evaluated analytically (J/m^3).
